In [1]:
import os
from databricks import sql
from dotenv import load_dotenv

In [2]:
# Carrega as variáveis de ambiente
load_dotenv()

server_hostname = os.getenv("DATABRICKS_SERVER_HOSTNAME")
http_path = os.getenv("DATABRICKS_HTTP_PATH")
access_token = os.getenv("DATABRICKS_TOKEN")

print("Ambiente configurado e pronto para a Camada Silver.")

Ambiente configurado e pronto para a Camada Silver.


In [3]:
with sql.connect(
    server_hostname=server_hostname,
    http_path=http_path,
    access_token=access_token
) as connection:
    with connection.cursor() as cursor:
        print("Iniciando a transformação e criação da camada Silver...")
        
        query_silver = """
        CREATE OR REPLACE TABLE cyclistic.silver.tripdata_clean AS
        WITH ranked_trips AS (
            SELECT 
                ride_id,
                rideable_type,
                try_to_timestamp(started_at) as started_at,
                try_to_timestamp(ended_at) as ended_at,
                start_station_name,
                start_station_id,
                end_station_name,
                end_station_id,
                start_lat,
                start_lng,
                end_lat,
                end_lng,
                member_casual,
                -- Identifica duplicidades pelo ride_id
                ROW_NUMBER() OVER (PARTITION BY ride_id ORDER BY started_at) AS rn
            FROM cyclistic.bronze.tripdata_raw
        )
        SELECT 
            ride_id,
            rideable_type,
            started_at,
            ended_at,
            start_station_name,
            start_station_id,
            end_station_name,
            end_station_id,
            start_lat,
            start_lng,
            end_lat,
            end_lng,
            member_casual
        FROM ranked_trips
        WHERE rn = 1
          -- Regra 1: Data final deve ser maior que data inicial
          AND ended_at > started_at
          -- Regra 2: Viagens com pelo menos 60 segundos (critério Divvy)
          AND (UNIX_TIMESTAMP(ended_at)) - UNIX_TIMESTAMP((started_at)) >= 60
          -- Regra 3: Viagens inferiores a 24 horas (opcional, evita anomalias extremas)
          AND (UNIX_TIMESTAMP(ended_at)) - UNIX_TIMESTAMP((started_at)) <= 86400
          -- Regra 4: Coordenadas válidas e não nulas
          AND start_lat IS NOT NULL AND start_lng IS NOT NULL
          AND end_lat IS NOT NULL AND end_lng IS NOT NULL
          AND start_lat != 0 AND start_lng != 0
          AND end_lat != 0 AND end_lng != 0
          -- Regra 5: ids válidos e não de estações nulos
          AND start_station_id IS NOT NULL AND end_station_id IS NOT NULL
          AND start_station_name IS NOT NULL AND end_station_name IS NOT NULL;
        """
        cursor.execute(query_silver)
        print("✅ Tabela 'cyclistic.silver.tripdata_clean' criada com sucesso!")

Iniciando a transformação e criação da camada Silver...
✅ Tabela 'cyclistic.silver.tripdata_clean' criada com sucesso!
